In [ ]:
import time
import datetime
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib widget

# Если у тебя есть pypylon
from pypylon import pylon

# Твои модули
from droplet_lib import DropletDetectorCuPy, detect_frame_to_dataset_layer

# (Опционально) если хочешь использовать готовые функции оффсетов
try:
    import Grid_measurer as gm
    precompute_disk_offsets = gm.precompute_disk_offsets
    sum_in_disk_u8 = gm.sum_in_disk_u8
except Exception:
    def precompute_disk_offsets(radius: int) -> np.ndarray:
        r = int(radius)
        ys, xs = np.ogrid[-r:r+1, -r:r+1]
        mask = (ys*ys + xs*xs) <= (r*r)
        dy, dx = np.where(mask)
        dy = dy - r
        dx = dx - r
        return np.stack([dy, dx], axis=1).astype(np.int16)

    def sum_in_disk_u8(img_u8: np.ndarray, cy: float, cx: float, offsets: np.ndarray) -> int:
        H, W = img_u8.shape
        y0 = int(round(cy))
        x0 = int(round(cx))
        yy = y0 + offsets[:, 0].astype(np.int32)
        xx = x0 + offsets[:, 1].astype(np.int32)
        m = (yy >= 0) & (yy < H) & (xx >= 0) & (xx < W)
        if not np.any(m):
            return 0
        return int(img_u8[yy[m], xx[m]].sum(dtype=np.int64))


def read_nth_frame_gray(video_path: str, frame_idx: int) -> np.ndarray:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")
    try:
        ok = cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        # ok может быть False на некоторых кодеках, но read всё равно часто работает
        ret, frame = cap.read()
        if not ret or frame is None:
            raise RuntimeError(f"Cannot read frame {frame_idx} from {video_path}")
        if frame.ndim == 3:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        else:
            gray = frame
        if gray.dtype != np.uint8:
            gray = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        return gray
    finally:
        cap.release()


def build_detector_from_widgets(w) -> DropletDetectorCuPy:
    # w = dict of widgets
    return DropletDetectorCuPy(
        percentile=float(w["percentile"].value),
        downscale_factor=int(w["ds"].value),
        droplet_radius_px=int(w["radius"].value),
        min_distance=int(w["min_dist"].value),
        cpu_downscale=bool(w["cpu_downscale"].value),
        sample_percentile_stride=int(w["stride"].value),
        max_candidates=int(w["max_candidates"].value),
        corr_backend=str(w["backend"].value),
        peak_downscale=int(w["peak_ds"].value),
        thr_mode=str(w["thr_mode"].value),
        thr_update_every=int(w["thr_every"].value),
        thr_ema_alpha=float(w["thr_alpha"].value),
        template_path=None,
        template_array=None,
        template_resize="auto",
    )


def detect_centers_on_frame(gray_u8: np.ndarray, detector: DropletDetectorCuPy):
    """
    Returns:
      xy (N,2) int32 in full-res pixels [x,y]
      scores (N,) float32
    """
    layer = detect_frame_to_dataset_layer(gray_u8, 0, detector)  # (N,4): frame,y,x,score
    if layer.shape[0] == 0:
        return np.zeros((0,2), np.int32), np.zeros((0,), np.float32)

    # full-res (x,y)
    xy = np.stack([layer[:, 2], layer[:, 1]], axis=1).astype(np.int32, copy=False)
    scores = layer[:, 3].astype(np.float32, copy=False)
    return xy, scores

In [ ]:
def make_camera_mono8():
    cam = pylon.InstantCamera(pylon.TlFactory.GetInstance().CreateFirstDevice())
    conv = pylon.ImageFormatConverter()
    conv.OutputPixelFormat = pylon.PixelType_Mono8
    conv.OutputBitAlignment = pylon.OutputBitAlignment_MsbAligned
    return cam, conv

def camera_configure(cam, fps=None, exposure_ms=None):
    cam.Open()
    if exposure_ms is not None:
        cam.ExposureTime.Value = float(exposure_ms) * 1000.0  # ms -> us
    if fps is not None:
        cam.AcquisitionFrameRateEnable.Value = True
        cam.AcquisitionFrameRate.Value = float(fps)

def record_intensity_fixed_centers_camera(
    out_tsv: Path,
    centers_xy: np.ndarray,   # (N,2) x,y in full-res pixels
    radius_px: int,
    duration_s: float,
    fps: float,
    exposure_ms: float,
    *,
    show_progress: bool = True,
):
    centers_xy = np.asarray(centers_xy, dtype=np.float32)
    n_ids = centers_xy.shape[0]
    if n_ids == 0:
        raise RuntimeError("centers_xy is empty")

    offsets = precompute_disk_offsets(int(radius_px))

    out_tsv = Path(out_tsv)
    out_tsv.parent.mkdir(parents=True, exist_ok=True)

    cam, conv = make_camera_mono8()
    camera_configure(cam, fps=fps, exposure_ms=exposure_ms)

    n_frames = int(round(duration_s * fps))
    header = ["t_s"] + [f"id{i+1:04d}" for i in range(n_ids)]

    t0_ns = None

    with open(out_tsv, "w", encoding="utf-8") as f:
        f.write("\t".join(header) + "\n")

        cam.StartGrabbing()
        try:
            for i in range(n_frames):
                grab = cam.RetrieveResult(5000, pylon.TimeoutHandling_ThrowException)
                try:
                    if not grab.GrabSucceeded():
                        continue
                    gray = conv.Convert(grab).GetArray()  # mono8
                    ts = int(grab.TimeStamp)              # ns-ish
                    if t0_ns is None:
                        t0_ns = ts
                    t_s = (ts - t0_ns) / 1e9

                    vals = []
                    for (x, y) in centers_xy:
                        vals.append(sum_in_disk_u8(gray, cy=y, cx=x, offsets=offsets))

                    f.write(f"{t_s:.6f}\t" + "\t".join(map(str, vals)) + "\n")

                finally:
                    grab.Release()

                if show_progress and (i % max(1, n_frames//10) == 0):
                    print(f"{i}/{n_frames}")

        finally:
            try:
                cam.StopGrabbing()
            except Exception:
                pass
            try:
                cam.Close()
            except Exception:
                pass

    return out_tsv


def record_intensity_fixed_centers_video(
    video_path: str,
    fps_manual: float,
    out_tsv: Path,
    centers_xy: np.ndarray,   # x,y in frame pixels
    radius_px: int,
    *,
    max_frames: int | None = None,
):
    centers_xy = np.asarray(centers_xy, dtype=np.float32)
    n_ids = centers_xy.shape[0]
    if n_ids == 0:
        raise RuntimeError("centers_xy is empty")

    offsets = precompute_disk_offsets(int(radius_px))
    out_tsv = Path(out_tsv)
    out_tsv.parent.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    header = ["t_s"] + [f"id{i+1:04d}" for i in range(n_ids)]

    with open(out_tsv, "w", encoding="utf-8") as f:
        f.write("\t".join(header) + "\n")

        fi = 0
        while True:
            ret, frame = cap.read()
            if not ret or frame is None:
                break
            if frame.ndim == 3:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            else:
                gray = frame
            if gray.dtype != np.uint8:
                gray = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

            t_s = fi / float(fps_manual)

            vals = []
            for (x, y) in centers_xy:
                vals.append(sum_in_disk_u8(gray, cy=y, cx=x, offsets=offsets))

            f.write(f"{t_s:.6f}\t" + "\t".join(map(str, vals)) + "\n")

            fi += 1
            if max_frames is not None and fi >= int(max_frames):
                break

    cap.release()
    return out_tsv

In [ ]:


# --- state ---
STATE = {
    "gray": None,
    "xy": None,        # (N,2) x,y
    "scores": None,    # (N,)
    "detector": None,
    "frame_idx": None,
    "video_path": None,
}

out = widgets.Output()

# --- widgets: input ---
video_w = widgets.Text("", description="Video path", layout=widgets.Layout(width="850px"))
frame_w = widgets.IntText(0, description="Frame N")

load_btn = widgets.Button(description="Load frame", button_style="info")
detect_btn = widgets.Button(description="Detect / Update overlay", button_style="success")

# --- detector parameter widgets ---
w = {
    "backend": widgets.Dropdown(options=["fft","ndimage"], value="fft", description="corr_backend"),
    "ds": widgets.IntText(4, description="ds"),
    "peak_ds": widgets.IntText(2, description="peak_ds"),
    "percentile": widgets.FloatText(88.0, description="percentile"),
    "min_dist": widgets.IntText(40, description="min_dist"),
    "radius": widgets.IntText(100, description="radius_px"),
    "cpu_downscale": widgets.Checkbox(True, description="CPU downscale"),
    "stride": widgets.IntText(1, description="pct_stride"),
    "max_candidates": widgets.IntText(20000, description="max_cand"),
    "thr_mode": widgets.Dropdown(options=["per_frame","cached","ema"], value="cached", description="thr_mode"),
    "thr_every": widgets.IntText(150, description="thr_every"),
    "thr_alpha": widgets.FloatText(0.1, description="thr_alpha"),
}

# display options
show_ids_w = widgets.Checkbox(False, description="Show IDs (1..N)")
min_score_w = widgets.FloatText(0.0, description="min_score")
topk_w = widgets.IntText(0, description="topK (0=all)")

# --- matplotlib figure ---
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.set_title("Frame + detections")
ax.axis("off")

_scatter = None
_texts = []

def _render():
    ax.clear()
    ax.axis("off")

    gray = STATE["gray"]
    if gray is None:
        ax.set_title("No frame loaded")
        fig.canvas.draw_idle()
        return

    ax.imshow(gray, cmap="gray")
    ax.set_title(f"Frame #{STATE['frame_idx']} | detections: {0 if STATE['xy'] is None else len(STATE['xy'])}")

    xy = STATE["xy"]
    scores = STATE["scores"]
    if xy is not None and len(xy) > 0:
        # plot points
        ax.scatter(xy[:,0], xy[:,1], s=20)

        if show_ids_w.value:
            # add small ID labels (can be heavy if many points)
            for i, (x, y) in enumerate(xy, start=1):
                ax.text(x+3, y-3, str(i), fontsize=6)

    fig.canvas.draw_idle()


def _log(msg):
    with out:
        print(msg)

def on_load(_):
    out.clear_output()
    vp = video_w.value.strip().strip('"')
    if not vp:
        _log("⚠️ empty video path")
        return
    n = int(frame_w.value)
    gray = read_nth_frame_gray(vp, n)
    STATE["gray"] = gray
    STATE["video_path"] = vp
    STATE["frame_idx"] = n
    STATE["xy"] = None
    STATE["scores"] = None
    _log(f"✅ loaded frame {n} | shape={gray.shape}")
    _render()

def on_detect(_):
    out.clear_output()
    if STATE["gray"] is None:
        _log("⚠️ load frame first")
        return

    det = build_detector_from_widgets(w)
    xy, scores = detect_centers_on_frame(STATE["gray"], det)

    # filters: min_score + topK
    if scores.size > 0:
        if float(min_score_w.value) > 0:
            m = scores >= float(min_score_w.value)
            xy = xy[m]
            scores = scores[m]
        if int(topk_w.value) > 0 and scores.size > int(topk_w.value):
            k = int(topk_w.value)
            idx = np.argpartition(scores, -k)[-k:]
            xy = xy[idx]
            scores = scores[idx]

    # sort by score (optional nice)
    if scores.size > 0:
        order = np.argsort(scores)[::-1]
        xy = xy[order]
        scores = scores[order]

    STATE["detector"] = det
    STATE["xy"] = xy.astype(np.int32, copy=False)
    STATE["scores"] = scores.astype(np.float32, copy=False)

    _log(f"✅ detections: {len(STATE['xy'])} | score range: {scores.min() if scores.size else None} .. {scores.max() if scores.size else None}")
    _render()

load_btn.on_click(on_load)
detect_btn.on_click(on_detect)

ui = widgets.VBox([
    widgets.HBox([video_w, frame_w, load_btn]),
    widgets.HBox([detect_btn, show_ids_w, min_score_w, topk_w]),
    widgets.HTML("<b>Detector params</b>"),
    widgets.HBox([w["backend"], w["ds"], w["peak_ds"], w["percentile"]]),
    widgets.HBox([w["min_dist"], w["radius"], w["cpu_downscale"], w["stride"], w["max_candidates"]]),
    widgets.HBox([w["thr_mode"], w["thr_every"], w["thr_alpha"]]),
    out
])

display(ui)
_render()

In [ ]:
import pandas as pd

# controls
src_w = widgets.Dropdown(options=["camera", "video"], value="camera", description="source")
fps_mfe_w = widgets.FloatText(20.0, description="FPS")
exp_mfe_w = widgets.FloatText(15.0, description="Exposure ms")
dur_mfe_w = widgets.FloatText(60.0, description="Duration s")
radius_mfe_w = widgets.IntText(100, description="Radius px")

video_fps_w = widgets.FloatText(20.0, description="Video FPS")  # manual

out_path_w = widgets.Text(str(Path("mfe_outputs") / "mfe_intensity.tsv"), description="out TSV", layout=widgets.Layout(width="850px"))
run_mfe_btn = widgets.Button(description="Start MFE intensity recording", button_style="danger")

plot_ids_w = widgets.Text("", description="plot IDs (e.g. 1,2,5) empty=all", layout=widgets.Layout(width="500px"))
plot_btn = widgets.Button(description="Plot I_id(t) + save PNG", button_style="success")
plot_png_w = widgets.Text(str(Path("mfe_outputs") / "I_id_t.png"), description="out PNG", layout=widgets.Layout(width="850px"))

mfe_out = widgets.Output()

def parse_ids(s: str, max_id: int):
    s = s.strip()
    if not s:
        return list(range(1, max_id+1))
    out = []
    for part in s.split(","):
        part = part.strip()
        if not part:
            continue
        out.append(int(part))
    out = [i for i in out if 1 <= i <= max_id]
    return out

def on_run_mfe(_):
    mfe_out.clear_output()
    if STATE["xy"] is None or len(STATE["xy"]) == 0:
        with mfe_out:
            print("⚠️ Нет детекций. Сначала Load frame и Detect.")
        return

    centers_xy = STATE["xy"]  # (N,2) x,y
    out_tsv = Path(out_path_w.value.strip().strip('"'))

    with mfe_out:
        print(f"Using centers: N={len(centers_xy)} | radius={radius_mfe_w.value}px | source={src_w.value}")

    if src_w.value == "camera":
        # камера: записываем duration_s * fps кадров
        out_file = record_intensity_fixed_centers_camera(
            out_tsv=out_tsv,
            centers_xy=centers_xy,
            radius_px=int(radius_mfe_w.value),
            duration_s=float(dur_mfe_w.value),
            fps=float(fps_mfe_w.value),
            exposure_ms=float(exp_mfe_w.value),
            show_progress=True
        )
        with mfe_out:
            print("✅ Saved:", out_file)

    else:
        vp = STATE["video_path"]
        if not vp:
            with mfe_out:
                print("⚠️ Нет video_path в STATE. Сначала Load frame.")
            return
        out_file = record_intensity_fixed_centers_video(
            video_path=vp,
            fps_manual=float(video_fps_w.value),
            out_tsv=out_tsv,
            centers_xy=centers_xy,
            radius_px=int(radius_mfe_w.value),
            max_frames=int(float(dur_mfe_w.value) * float(video_fps_w.value))
        )
        with mfe_out:
            print("✅ Saved:", out_file)

def on_plot(_):
    mfe_out.clear_output()
    out_tsv = Path(out_path_w.value.strip().strip('"'))
    if not out_tsv.exists():
        with mfe_out:
            print("⚠️ TSV not found:", out_tsv)
        return

    df = pd.read_csv(out_tsv, sep="\t")
    cols = [c for c in df.columns if c.startswith("id")]
    if not cols:
        with mfe_out:
            print("⚠️ No id columns in TSV")
        return

    max_id = len(cols)
    ids = parse_ids(plot_ids_w.value, max_id=max_id)

    plt.figure(figsize=(10,5))
    for i in ids:
        col = f"id{i:04d}"
        if col in df.columns:
            plt.plot(df["t_s"].to_numpy(), df[col].to_numpy(), label=col)

    plt.xlabel("t (s)")
    plt.ylabel("Integrated intensity in disk (a.u.)")
    plt.title("I_id(t) for detected droplets (fixed centers)")
    plt.grid(True, alpha=0.3)
    if len(ids) <= 12:
        plt.legend()
    plt.tight_layout()

    out_png = Path(plot_png_w.value.strip().strip('"'))
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()

    with mfe_out:
        print("✅ Saved plot:", out_png)

run_mfe_btn.on_click(on_run_mfe)
plot_btn.on_click(on_plot)

display(widgets.VBox([
    widgets.HTML("<b>MFE intensity recording (fixed centers)</b>"),
    widgets.HBox([src_w, fps_mfe_w, exp_mfe_w, dur_mfe_w]),
    widgets.HBox([radius_mfe_w, video_fps_w]),
    out_path_w,
    run_mfe_btn,
    widgets.HTML("<b>Plot</b>"),
    widgets.HBox([plot_ids_w, plot_btn]),
    plot_png_w,
    mfe_out
]))

In [ ]:
import numpy as np

def fit_exp_fast(t, y):
    """I(t) ≈ C + A exp(-k t) (быстрый линейный фит в log-домене)."""
    t = np.asarray(t, np.float64)
    y = np.asarray(y, np.float64)

    C = float(np.percentile(y[np.isfinite(y)], 5))
    z = y - C
    eps = 1e-6 * max(1.0, float(np.nanmax(y) - np.nanmin(y)))
    z = np.clip(z, eps, None)

    p = np.polyfit(t, np.log(z), deg=1)
    k = float(-p[0])
    A = float(np.exp(p[1]))
    fitted = C + A * np.exp(-k * t)
    return fitted, (C, A, k)

def fit_baseline_exp_robust(t, y, iters=3, keep_quantile=0.6):
    """
    Робастно оценивает baseline B(t) при наличии провалов (ON окон), которые только уменьшают сигнал.
    keep_quantile=0.6 -> оставляем верхние 40% по отношению y/fitted.
    """
    t = np.asarray(t, np.float64)
    y = np.asarray(y, np.float64)
    m = np.isfinite(t) & np.isfinite(y)
    t = t[m]; y = y[m]

    fitted, params = fit_exp_fast(t, y)

    for _ in range(int(iters)):
        r = y / (fitted + 1e-12)
        thr = np.quantile(r, float(keep_quantile))
        keep = r >= thr
        if np.sum(keep) < 20:
            break
        fitted, params = fit_exp_fast(t[keep], y[keep])
        # пересчёт на всей сетке t
        C, A, k = params
        fitted = C + A * np.exp(-k * t)

    return fitted, params, t, y

In [ ]:
def make_pulse_template(n, fs, N_on_s, off_factor=1.5):
    """
    Periodic template length n samples.
    tpl = 1 during ON (N seconds), 0 during OFF (1.5N seconds).
    """
    fs = float(fs)
    Lon = max(1, int(round(float(N_on_s) * fs)))
    Loff = max(1, int(round(float(off_factor) * float(N_on_s) * fs)))
    Lp = Lon + Loff

    one = np.concatenate([np.ones(Lon), np.zeros(Loff)]).astype(np.float64)
    reps = int(np.ceil(n / Lp))
    tpl = np.tile(one, reps)[:n]
    return tpl, Lon, Loff, Lp

def best_shift_scale_corr(y, tpl):
    """
    Circular matched filter via FFT to get best phase shift.
    Then LS scale to estimate amplitude.
    """
    y = np.asarray(y, np.float64)
    tpl = np.asarray(tpl, np.float64)
    n = y.size

    y0 = y - np.mean(y)
    t0 = tpl - np.mean(tpl)

    Y = np.fft.rfft(y0)
    T = np.fft.rfft(t0)
    corr = np.fft.irfft(Y * np.conj(T), n=n)

    shift = int(np.argmax(corr))
    t_shift = np.roll(t0, shift)

    denom = float(np.dot(t_shift, t_shift)) + 1e-12
    a = float(np.dot(y0, t_shift) / denom)

    corr_norm = float(np.dot(y0, t_shift) / (np.linalg.norm(y0)*np.linalg.norm(t_shift) + 1e-12))
    return shift, a, corr_norm

def search_pulse_cycle(r, fs, N_candidates_s, off_factor=1.5):
    """
    r(t) ~ -a * tpl(t) + noise
    ищем N и фазу по максимальной нормированной корреляции.
    """
    r = np.asarray(r, np.float64)
    best = None

    for N_on_s in N_candidates_s:
        tpl, Lon, Loff, Lp = make_pulse_template(len(r), fs, N_on_s, off_factor=off_factor)
        shift, a, c = best_shift_scale_corr(r, tpl)

        cand = dict(N_on_s=float(N_on_s), shift=int(shift), scale=float(a), corr=float(c),
                    Lon=int(Lon), Loff=int(Loff), Lp=int(Lp))
        if (best is None) or (cand["corr"] > best["corr"]):
            best = cand
    return best

In [ ]:
def deltaI_over_I_pulse(y_corr, shift_in_period, Lon, Lp):
    """
    y_corr ~ 1 (OFF) and ~ 1-a (ON).
    shift_in_period: start of period
    """
    y_corr = np.asarray(y_corr, np.float64)
    n = y_corr.size
    idx = (np.arange(n, dtype=np.int64) - int(shift_in_period)) % int(Lp)

    on = idx < int(Lon)
    off = ~on

    if not (np.any(on) and np.any(off)):
        return np.nan, np.nan, np.nan

    I_on  = float(np.median(y_corr[on]))
    I_off = float(np.median(y_corr[off]))

    d = I_off - I_on
    I0 = 0.5 * (I_off + I_on)
    return d/(I0 + 1e-12), I_on, I_off

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# 1) Uniform resampling (FFT/matched filter assumes uniform grid)
# ------------------------------------------------------------
def _resample_uniform(t, y, target_fs=None):
    t = np.asarray(t, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    m = np.isfinite(t) & np.isfinite(y)
    t = t[m]; y = y[m]
    if t.size < 20:
        return None, None, None

    order = np.argsort(t)
    t = t[order]; y = y[order]
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if dt.size == 0:
        return None, None, None

    dt_med = float(np.median(dt))
    fs = (1.0 / dt_med) if target_fs is None else float(target_fs)
    dt_u = 1.0 / fs

    t0 = float(t[0]); t1 = float(t[-1])
    n = int(np.floor((t1 - t0) / dt_u)) + 1
    if n < 32:
        return None, None, None

    t_u = t0 + dt_u*np.arange(n, dtype=np.float64)
    y_u = np.interp(t_u, t, y).astype(np.float64, copy=False)
    return t_u, y_u, fs


# ------------------------------------------------------------
# 2) Robust baseline fit: exponential bleaching B(t)
#    (drops only go DOWN during ON; baseline is "upper envelope")
# ------------------------------------------------------------
def fit_baseline_exp_robust(t, y, iters=3, keep_quantile=0.6):
    """
    Iteratively fit exp to upper part of y relative to current fit.
    keep_quantile=0.6 -> keep top 40% points by y/fitted.
    """
    t = np.asarray(t, np.float64)
    y = np.asarray(y, np.float64)

    fitted, params = fit_exp_fast(t, y)  # <-- твоё уже существующее

    for _ in range(int(iters)):
        r = y / (fitted + 1e-12)
        thr = float(np.quantile(r[np.isfinite(r)], float(keep_quantile)))
        keep = r >= thr
        if np.sum(keep) < 30:
            break
        fitted_k, params_k = fit_exp_fast(t[keep], y[keep])
        # rebuild on full t
        C, A, k = params_k
        fitted = C + A*np.exp(-k*t)
        params = params_k

    return fitted, params


# ------------------------------------------------------------
# 3) Remove slow drift + notch at ~90 s period (optional but recommended)
#    Apply to residual r(t) = y_corr - 1 (not to raw intensity)
# ------------------------------------------------------------
def spectral_hp_notch(x, fs, hp_cut_hz=0.02, notch_period_s=90.0, notch_bw_hz=0.002, notch_harmonics=1):
    x = np.asarray(x, dtype=np.float64)
    n = x.size
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(n, d=1.0/fs)

    # High-pass
    X[f < float(hp_cut_hz)] = 0.0

    # Notch at 1/90 Hz (+ harmonics if needed)
    f0 = 1.0/float(notch_period_s)
    bw = float(notch_bw_hz)
    for h in range(1, int(notch_harmonics)+1):
        fc = h*f0
        X[np.abs(f - fc) <= bw] = 0.0

    return np.fft.irfft(X, n=n)


# ------------------------------------------------------------
# 4) Pulse template: ON for N seconds, OFF for 1.5N seconds
#    Matched filter to find best N and phase
# ------------------------------------------------------------
def make_pulse_template(n, fs, N_on_s, off_factor=1.5):
    fs = float(fs)
    Lon = max(1, int(round(float(N_on_s) * fs)))
    Loff = max(1, int(round(float(off_factor) * float(N_on_s) * fs)))
    Lp = Lon + Loff

    one = np.concatenate([np.ones(Lon), np.zeros(Loff)]).astype(np.float64)
    reps = int(np.ceil(n / Lp))
    tpl = np.tile(one, reps)[:n]
    return tpl, Lon, Loff, Lp

def best_shift_scale_corr(y, tpl):
    """
    Circular correlation via FFT -> best shift.
    Then LS scale.
    """
    y = np.asarray(y, np.float64)
    tpl = np.asarray(tpl, np.float64)
    n = y.size

    y0 = y - np.mean(y)
    t0 = tpl - np.mean(tpl)

    Y = np.fft.rfft(y0)
    T = np.fft.rfft(t0)
    corr = np.fft.irfft(Y * np.conj(T), n=n)

    shift = int(np.argmax(corr))
    t_shift = np.roll(t0, shift)

    denom = float(np.dot(t_shift, t_shift)) + 1e-12
    a = float(np.dot(y0, t_shift) / denom)

    corr_norm = float(np.dot(y0, t_shift) / (np.linalg.norm(y0)*np.linalg.norm(t_shift) + 1e-12))
    return shift, a, corr_norm

def search_pulse_cycle(r_hp, fs, N_candidates_s, off_factor=1.5):
    """
    r_hp ~ -a * tpl + noise
    (negative dips during ON), but scale can handle sign.
    """
    r_hp = np.asarray(r_hp, np.float64)
    best = None
    for N_on_s in N_candidates_s:
        tpl, Lon, Loff, Lp = make_pulse_template(len(r_hp), fs, N_on_s, off_factor=off_factor)
        shift, a, c = best_shift_scale_corr(r_hp, tpl)
        cand = dict(N_on_s=float(N_on_s), shift=int(shift), scale=float(a), corr=float(c),
                    Lon=int(Lon), Loff=int(Loff), Lp=int(Lp))
        if (best is None) or (cand["corr"] > best["corr"]):
            best = cand
    return best


# ------------------------------------------------------------
# 5) Main processing for TSV (drop-in replacement)
# ------------------------------------------------------------
def process_mfe_tsv_pulse(
    tsv_path: str,
    out_dir: str,
    *,
    off_factor: float = 1.5,

    # N search: either pass N_on_s (known) OR search range
    N_on_s: float | None = None,
    N_min_s: float = 0.4,
    N_max_s: float = 2.0,
    N_steps: int = 21,

    # baseline robust fit
    baseline_iters: int = 3,
    baseline_keep_q: float = 0.6,

    # drift removal on residual
    do_hp_notch: bool = True,
    hp_cut_hz: float = 0.02,
    notch_period_s: float = 90.0,
    notch_bw_hz: float = 0.002,
    notch_harmonics: int = 1,

    # outputs
    save_processed_tsv: bool = True,
    plot_ids: list[int] | None = None,    # e.g. [1,3,5] to save diagnostics
):
    """
    Input TSV:
      t_s  id0001 id0002 ...

    Output in out_dir:
      summary.csv    (per id: N, period, corr, snr, deltaI/I, etc.)
      processed.tsv  (t_s + *_corr + *_rhp + *_recon)
      diagnostics for selected plot_ids:
        idXXXX_diag.png
    """
    tsv_path = Path(tsv_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(tsv_path, sep="\t")
    if "t_s" not in df.columns:
        raise RuntimeError("TSV must contain 't_s' column")
    t_raw = df["t_s"].to_numpy(np.float64)
    id_cols = [c for c in df.columns if c.startswith("id")]
    if not id_cols:
        raise RuntimeError("No idXXXX columns found in TSV")

    # build uniform grid once (fs derived from data)
    t_u, y0_u, fs = _resample_uniform(t_raw, df[id_cols[0]].to_numpy(np.float64))
    if t_u is None:
        raise RuntimeError("Failed to resample to uniform grid (check timestamps/NaNs).")

    # N candidate list
    if N_on_s is None:
        N_candidates = np.linspace(float(N_min_s), float(N_max_s), int(N_steps)).tolist()
    else:
        N_candidates = [float(N_on_s)]

    # which IDs to plot
    want_plot = set()
    if plot_ids is not None:
        for i in plot_ids:
            want_plot.add(f"id{i:04d}")

    summary = []
    processed = {"t_s": t_u}

    for col in id_cols:
        # resample this series
        _, y_u, _ = _resample_uniform(t_raw, df[col].to_numpy(np.float64), target_fs=fs)
        if y_u is None:
            continue

        # robust baseline bleaching fit
        B_u, (C, A, k) = fit_baseline_exp_robust(
            t_u, y_u, iters=baseline_iters, keep_quantile=baseline_keep_q
        )
        y_corr = y_u / (B_u + 1e-12)      # ~1 on OFF
        r = y_corr - 1.0                 # dips negative on ON

        # drift removal on residual
        if do_hp_notch:
            r_hp = spectral_hp_notch(
                r - np.mean(r), fs,
                hp_cut_hz=hp_cut_hz,
                notch_period_s=notch_period_s,
                notch_bw_hz=notch_bw_hz,
                notch_harmonics=notch_harmonics
            )
        else:
            r_hp = (r - np.mean(r))

        # matched filter search (N + phase)
        best = search_pulse_cycle(r_hp, fs, N_candidates, off_factor=off_factor)
        if best is None:
            summary.append({
                "id": col, "status": "no_match",
                "N_on_s": np.nan, "period_s": np.nan,
                "corr": np.nan, "snr": np.nan,
                "deltaI_over_I": np.nan, "deltaI_over_I_pct": np.nan,
                "exp_k": k,
            })
            processed[f"{col}_corr"] = y_corr
            processed[f"{col}_rhp"] = r_hp
            continue

        # reconstruct fitted pulse component (in residual domain)
        tpl, Lon, Loff, Lp = make_pulse_template(len(r_hp), fs, best["N_on_s"], off_factor=off_factor)
        tpl0 = tpl - np.mean(tpl)
        tpl_shift = np.roll(tpl0, best["shift"])
        recon = best["scale"] * tpl_shift

        # SNR in residual domain
        sig_rms = float(np.sqrt(np.mean(recon**2)))
        noise_rms = float(np.sqrt(np.mean((r_hp - recon)**2)))
        snr = sig_rms / (noise_rms + 1e-12)

        # ΔI/I in corrected domain (ON vs OFF)
        shift_in_period = int(best["shift"] % int(Lp))
        dI_over_I, I_on, I_off = deltaI_over_I_pulse(y_corr, shift_in_period, Lon, Lp)

        summary.append({
            "id": col, "status": "ok",
            "fs_hz": fs,
            "N_on_s": float(best["N_on_s"]),
            "period_s": float((1.0 + off_factor) * best["N_on_s"]),  # 2.5N
            "Lon": int(Lon), "Loff": int(Loff), "Lp": int(Lp),
            "shift_samples": int(best["shift"]),
            "corr": float(best["corr"]),
            "scale": float(best["scale"]),
            "snr": float(snr),
            "sig_rms": float(sig_rms),
            "noise_rms": float(noise_rms),
            "deltaI_over_I": float(dI_over_I),
            "deltaI_over_I_pct": float(100.0*dI_over_I),
            "I_on_med": float(I_on),
            "I_off_med": float(I_off),
            "exp_k": float(k),
        })

        processed[f"{col}_corr"] = y_corr
        processed[f"{col}_rhp"] = r_hp
        processed[f"{col}_recon"] = recon

        # diagnostics plot
        if col in want_plot:
            # build ON mask for visualization
            idx = (np.arange(len(t_u), dtype=np.int64) - shift_in_period) % int(Lp)
            on_mask = idx < int(Lon)

            plt.figure(figsize=(11,4.5))
            plt.plot(t_u, y_corr, label="I / baseline")
            # highlight ON windows
            # (draw as translucent vertical spans)
            # to avoid many spans, mark transitions
            in_on = False
            start = None
            for i in range(len(on_mask)):
                if on_mask[i] and not in_on:
                    in_on = True
                    start = t_u[i]
                elif (not on_mask[i]) and in_on:
                    in_on = False
                    end = t_u[i]
                    plt.axvspan(start, end, alpha=0.15)
            if in_on and start is not None:
                plt.axvspan(start, t_u[-1], alpha=0.15)

            plt.xlabel("t (s)")
            plt.ylabel("Normalized intensity")
            plt.title(f"{col} | N={best['N_on_s']:.3g}s | T={((1+off_factor)*best['N_on_s']):.3g}s | ΔI/I={100*dI_over_I:.3g}% | corr={best['corr']:.3g}")
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.tight_layout()
            plt.savefig(out_dir / f"{col}_diag.png", dpi=300, bbox_inches="tight")
            plt.show()

    df_sum = pd.DataFrame(summary)
    df_sum.to_csv(out_dir / "summary.csv", index=False)

    if save_processed_tsv:
        pd.DataFrame(processed).to_csv(out_dir / "processed.tsv", sep="\t", index=False)

    print("✅ saved:", out_dir / "summary.csv")
    if save_processed_tsv:
        print("✅ saved:", out_dir / "processed.tsv")

    return df_sum

In [ ]:
out_tsv = r"data\MFE_pro_size\mfe_3px.tsv"
out_dir = r"data\MFE_pro_size\out3"

In [ ]:
df_sum = process_mfe_tsv_pulse(
    tsv_path=out_tsv,
    out_dir=out_dir,
    N_on_s=2,          # фиксируем N
    do_hp_notch=True,
)

In [ ]:
import re, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -----------------------------
# 0) Utilities
# -----------------------------
def parse_size_um_from_path(path: str):
    """
    Tries to parse droplet size from filename/folder.
    Recognizes patterns like: 80um, 80_um, d80um, diam80um, 80µm.
    Returns float or None.
    """
    s = str(path)
    s = s.replace("µ", "u")
    m = re.search(r"(?i)(?:diam|d|size)?[_-]?(\d+(?:\.\d+)?)\s*u?m", s)
    if m:
        return float(m.group(1))
    return None


def resample_uniform(t, y, target_fs=None):
    t = np.asarray(t, np.float64)
    y = np.asarray(y, np.float64)
    m = np.isfinite(t) & np.isfinite(y)
    t = t[m]; y = y[m]
    if t.size < 20:
        return None, None, None
    order = np.argsort(t)
    t = t[order]; y = y[order]
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if dt.size == 0:
        return None, None, None
    dt_med = float(np.median(dt))
    fs = (1.0/dt_med) if target_fs is None else float(target_fs)
    dt_u = 1.0/fs
    t0 = float(t[0]); t1 = float(t[-1])
    n = int(np.floor((t1 - t0)/dt_u)) + 1
    if n < 32:
        return None, None, None
    t_u = t0 + dt_u*np.arange(n, dtype=np.float64)
    y_u = np.interp(t_u, t, y).astype(np.float64, copy=False)
    return t_u, y_u, fs


# -----------------------------
# 1) Baseline: exponential bleaching
# -----------------------------
def fit_exp_fast(t, y):
    """
    Fast fit: I(t) ≈ C + A exp(-k t) via log-linear on (I-C).
    """
    t = np.asarray(t, np.float64)
    y = np.asarray(y, np.float64)
    C = float(np.percentile(y[np.isfinite(y)], 5))
    z = y - C
    eps = 1e-6 * max(1.0, float(np.nanmax(y) - np.nanmin(y)))
    z = np.clip(z, eps, None)
    p = np.polyfit(t, np.log(z), deg=1)
    k = float(-p[0])
    A = float(np.exp(p[1]))
    fitted = C + A*np.exp(-k*t)
    return fitted, (C, A, k)


def fit_baseline_exp_robust(t, y, iters=3, keep_quantile=0.6):
    """
    Robust baseline for case when modulation only produces DOWN dips.
    Iteratively fits exp to upper envelope (keeps top fraction by y/fitted).
    """
    t = np.asarray(t, np.float64)
    y = np.asarray(y, np.float64)

    fitted, params = fit_exp_fast(t, y)
    for _ in range(int(iters)):
        r = y / (fitted + 1e-12)
        thr = float(np.quantile(r[np.isfinite(r)], float(keep_quantile)))
        keep = r >= thr
        if np.sum(keep) < 30:
            break
        _, params_k = fit_exp_fast(t[keep], y[keep])
        C, A, k = params_k
        fitted = C + A*np.exp(-k*t)
        params = params_k
    return fitted, params


# -----------------------------
# 2) Drift removal on residual: high-pass + notch @ 90s
# -----------------------------
def spectral_hp_notch(x, fs, hp_cut_hz=0.02, notch_period_s=90.0, notch_bw_hz=0.002, notch_harmonics=1):
    x = np.asarray(x, np.float64)
    n = x.size
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(n, d=1.0/fs)

    X[f < float(hp_cut_hz)] = 0.0

    f0 = 1.0/float(notch_period_s)
    bw = float(notch_bw_hz)
    for h in range(1, int(notch_harmonics)+1):
        fc = h*f0
        X[np.abs(f - fc) <= bw] = 0.0

    return np.fft.irfft(X, n=n)


# -----------------------------
# 3) Pulse pattern: ON=N, OFF=1.5N, matched filter
# -----------------------------
def make_pulse_template(n, fs, N_on_s, off_factor=1.5):
    fs = float(fs)
    Lon = max(1, int(round(float(N_on_s)*fs)))
    Loff = max(1, int(round(float(off_factor)*float(N_on_s)*fs)))
    Lp = Lon + Loff
    one = np.concatenate([np.ones(Lon), np.zeros(Loff)]).astype(np.float64)
    reps = int(np.ceil(n / Lp))
    tpl = np.tile(one, reps)[:n]
    return tpl, Lon, Loff, Lp


def best_shift_scale_corr(y, tpl):
    """
    Circular correlation (FFT) -> best phase shift.
    Then LS scale.
    """
    y = np.asarray(y, np.float64)
    tpl = np.asarray(tpl, np.float64)
    n = y.size

    y0 = y - np.mean(y)
    t0 = tpl - np.mean(tpl)

    Y = np.fft.rfft(y0)
    T = np.fft.rfft(t0)
    corr = np.fft.irfft(Y * np.conj(T), n=n)

    shift = int(np.argmax(corr))
    t_shift = np.roll(t0, shift)

    denom = float(np.dot(t_shift, t_shift)) + 1e-12
    a = float(np.dot(y0, t_shift) / denom)

    corr_norm = float(np.dot(y0, t_shift) / (np.linalg.norm(y0)*np.linalg.norm(t_shift) + 1e-12))
    return shift, a, corr_norm


def search_pulse_cycle(r_hp, fs, N_candidates_s, off_factor=1.5):
    best = None
    for N_on_s in N_candidates_s:
        tpl, Lon, Loff, Lp = make_pulse_template(len(r_hp), fs, N_on_s, off_factor=off_factor)
        shift, scale, c = best_shift_scale_corr(r_hp, tpl)
        cand = dict(N_on_s=float(N_on_s), shift=int(shift), scale=float(scale), corr=float(c),
                    Lon=int(Lon), Loff=int(Loff), Lp=int(Lp))
        if (best is None) or (cand["corr"] > best["corr"]):
            best = cand
    return best


def deltaI_over_I_pulse(y_corr, shift_in_period, Lon, Lp):
    """
    ΔI/I computed from corrected signal:
      OFF: y_corr ~ 1
      ON : y_corr ~ 1-a
    Uses median OFF vs median ON.
    """
    y_corr = np.asarray(y_corr, np.float64)
    n = y_corr.size
    idx = (np.arange(n, dtype=np.int64) - int(shift_in_period)) % int(Lp)

    on = idx < int(Lon)
    off = ~on
    if not (np.any(on) and np.any(off)):
        return np.nan, np.nan, np.nan

    I_on = float(np.median(y_corr[on]))
    I_off = float(np.median(y_corr[off]))
    d = I_off - I_on
    I0 = 0.5*(I_off + I_on)
    return d/(I0 + 1e-12), I_on, I_off


# -----------------------------
# 4) Process ONE TSV -> per-droplet table
# -----------------------------
def process_one_raw_tsv(
    tsv_path: str,
    size_um: float | None,
    *,
    label: str | None = None,
    off_factor: float = 1.5,
    N_on_s: float | None = None,       # if known -> FIX
    N_min_s: float = 0.4,
    N_max_s: float = 2.0,
    N_steps: int = 21,
    baseline_iters: int = 3,
    baseline_keep_q: float = 0.6,
    do_hp_notch: bool = True,
    hp_cut_hz: float = 0.02,
    notch_period_s: float = 90.0,
    notch_bw_hz: float = 0.002,
    notch_harmonics: int = 1,
):
    df = pd.read_csv(tsv_path, sep="\t")
    if "t_s" not in df.columns:
        raise RuntimeError(f"{tsv_path}: no t_s column")
    t_raw = df["t_s"].to_numpy(np.float64)
    id_cols = [c for c in df.columns if c.startswith("id")]
    if not id_cols:
        raise RuntimeError(f"{tsv_path}: no idXXXX columns")

    # uniform grid from first id
    t_u, y0_u, fs = resample_uniform(t_raw, df[id_cols[0]].to_numpy(np.float64))
    if t_u is None:
        raise RuntimeError(f"{tsv_path}: cannot resample to uniform grid")

    # N candidates
    if N_on_s is None:
        N_candidates = np.linspace(float(N_min_s), float(N_max_s), int(N_steps)).tolist()
    else:
        N_candidates = [float(N_on_s)]

    rows = []
    for col in id_cols:
        _, y_u, _ = resample_uniform(t_raw, df[col].to_numpy(np.float64), target_fs=fs)
        if y_u is None:
            continue

        # baseline
        B_u, (_, _, k) = fit_baseline_exp_robust(t_u, y_u, iters=baseline_iters, keep_quantile=baseline_keep_q)
        y_corr = y_u / (B_u + 1e-12)
        r = y_corr - 1.0

        # drift removal on residual
        if do_hp_notch:
            r_hp = spectral_hp_notch(r - np.mean(r), fs,
                                     hp_cut_hz=hp_cut_hz,
                                     notch_period_s=notch_period_s,
                                     notch_bw_hz=notch_bw_hz,
                                     notch_harmonics=notch_harmonics)
        else:
            r_hp = (r - np.mean(r))

        # matched filter (N + phase)
        best = search_pulse_cycle(r_hp, fs, N_candidates, off_factor=off_factor)
        if best is None:
            continue

        # reconstruct fitted pulse (for SNR)
        tpl, Lon, Loff, Lp = make_pulse_template(len(r_hp), fs, best["N_on_s"], off_factor=off_factor)
        tpl0 = tpl - np.mean(tpl)
        recon = best["scale"] * np.roll(tpl0, best["shift"])

        sig_rms = float(np.sqrt(np.mean(recon**2)))
        noise_rms = float(np.sqrt(np.mean((r_hp - recon)**2)))
        snr = sig_rms / (noise_rms + 1e-12)
        snr_db = 20.0*np.log10(snr + 1e-12)

        # ΔI/I (ON vs OFF in corrected domain)
        shift_in_period = int(best["shift"] % int(Lp))
        dI_over_I, I_on, I_off = deltaI_over_I_pulse(y_corr, shift_in_period, Lon, Lp)

        rows.append({
            "source_tsv": str(tsv_path),
            "label": (label if label is not None else Path(tsv_path).stem),
            "size_um": float(size_um) if size_um is not None else np.nan,
            "id": col,
            "fs_hz": float(fs),
            "N_on_s": float(best["N_on_s"]),
            "period_s": float((1.0 + off_factor) * best["N_on_s"]),
            "corr": float(best["corr"]),
            "snr": float(snr),
            "snr_db": float(snr_db),
            "sig_rms": float(sig_rms),
            "noise_rms": float(noise_rms),
            "deltaI_over_I": float(dI_over_I),
            "deltaI_over_I_pct": float(100.0 * dI_over_I),
            "I_on_med": float(I_on),
            "I_off_med": float(I_off),
            "exp_k": float(k),
            "median_I_raw": float(np.median(y_u)),
        })

    return pd.DataFrame(rows)


# -----------------------------
# 5) Batch: many TSVs -> merged + aggregate + plots
# -----------------------------
def batch_size_dependency(
    tsv_glob: str,
    out_dir: str,
    *,
    size_map: dict[str, float] | None = None,   # optional: {"small":80, "medium":120, ...} matched by substring
    N_on_s: float | None = None,                # if known, set here (recommended!)
    N_min_s: float = 0.4,
    N_max_s: float = 2.0,
    N_steps: int = 21,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    paths = [Path(p) for p in glob.glob(tsv_glob, recursive=True)]
    if not paths:
        raise RuntimeError("No TSV files found by glob")

    all_rows = []
    for p in paths:
        # size inference
        size_um = parse_size_um_from_path(str(p))
        label = p.parent.name  # handy label
        if size_map is not None:
            # if any key substring matches path -> use mapped size
            sp = str(p).lower()
            for k, v in size_map.items():
                if k.lower() in sp:
                    size_um = float(v)
                    break

        df_one = process_one_raw_tsv(
            tsv_path=str(p),
            size_um=size_um,
            label=label,
            N_on_s=N_on_s,
            N_min_s=N_min_s,
            N_max_s=N_max_s,
            N_steps=N_steps,
        )
        if len(df_one):
            all_rows.append(df_one)

    if not all_rows:
        raise RuntimeError("Processed 0 droplets (check TSV format / parameters)")

    df_all = pd.concat(all_rows, ignore_index=True)
    df_all.to_csv(out_dir / "per_droplet.csv", index=False)

    # aggregate by size (numeric)
    df_num = df_all[np.isfinite(df_all["size_um"])].copy()
    if len(df_num) == 0:
        # fallback: categorical by label
        df_num = df_all.copy()
        df_num["size_um"] = df_num["label"].astype(str)

    # group stats: median + IQR
    def agg(series):
        x = series.to_numpy(np.float64)
        x = x[np.isfinite(x)]
        if x.size == 0:
            return pd.Series({"n":0, "median":np.nan, "p25":np.nan, "p75":np.nan, "mean":np.nan, "std":np.nan})
        return pd.Series({
            "n": int(x.size),
            "median": float(np.median(x)),
            "p25": float(np.quantile(x, 0.25)),
            "p75": float(np.quantile(x, 0.75)),
            "mean": float(np.mean(x)),
            "std": float(np.std(x)),
        })

    g = df_num.groupby("size_um", sort=True)

    agg_delta = g["deltaI_over_I_pct"].apply(agg).reset_index()
    agg_snrdb = g["snr_db"].apply(agg).reset_index()
    agg_snr   = g["snr"].apply(agg).reset_index()

    agg_delta.to_csv(out_dir / "per_size_deltaI.csv", index=False)
    agg_snrdb.to_csv(out_dir / "per_size_snr_db.csv", index=False)
    agg_snr.to_csv(out_dir / "per_size_snr.csv", index=False)

    # ---- PLOTS ----
    # If numeric sizes: use errorbars; else boxplots.
    is_numeric_size = np.issubdtype(df_num["size_um"].dtype, np.number)

    if is_numeric_size:
        # ΔI/I vs size (median with IQR)
        xs = agg_delta["size_um"].to_numpy(np.float64)
        med = agg_delta["median"].to_numpy(np.float64)
        yerr_lo = med - agg_delta["p25"].to_numpy(np.float64)
        yerr_hi = agg_delta["p75"].to_numpy(np.float64) - med

        plt.figure(figsize=(8,5))
        plt.errorbar(xs, med, yerr=[yerr_lo, yerr_hi], fmt="o-")
        plt.xlabel("Droplet size (µm)")
        plt.ylabel("ΔI/I (%)")
        plt.title("ΔI/I vs droplet size (median ± IQR)")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(out_dir / "deltaI_vs_size.png", dpi=100, bbox_inches="tight")
        plt.show()

        # SNR(dB) vs size
        xs = agg_snrdb["size_um"].to_numpy(np.float64)
        med = agg_snrdb["median"].to_numpy(np.float64)
        yerr_lo = med - agg_snrdb["p25"].to_numpy(np.float64)
        yerr_hi = agg_snrdb["p75"].to_numpy(np.float64) - med

        plt.figure(figsize=(8,5))
        plt.errorbar(xs, med, yerr=[yerr_lo, yerr_hi], fmt="o-")
        plt.xlabel("Droplet size (µm)")
        plt.ylabel("SNR (dB)")
        plt.title("SNR vs droplet size (median ± IQR)")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(out_dir / "snr_db_vs_size.png", dpi=100, bbox_inches="tight")
        plt.show()

    else:
        # categorical boxplots
        order = agg_delta["size_um"].astype(str).tolist()

        plt.figure(figsize=(9,5))
        data = [df_num[df_num["size_um"].astype(str)==str(s)]["deltaI_over_I_pct"].to_numpy(np.float64) for s in order]
        plt.boxplot(data, labels=order, showfliers=False)
        plt.xlabel("Droplet size group")
        plt.ylabel("ΔI/I (%)")
        plt.title("ΔI/I by droplet size group")
        plt.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(out_dir / "deltaI_box_by_group.png", dpi=100, bbox_inches="tight")
        plt.show()

        plt.figure(figsize=(9,5))
        data = [df_num[df_num["size_um"].astype(str)==str(s)]["snr_db"].to_numpy(np.float64) for s in order]
        plt.boxplot(data, labels=order, showfliers=False)
        plt.xlabel("Droplet size group")
        plt.ylabel("SNR (dB)")
        plt.title("SNR by droplet size group")
        plt.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(out_dir / "snr_db_box_by_group.png", dpi=100, bbox_inches="tight")
        plt.show()

    print("✅ Saved:")
    print(" ", out_dir / "per_droplet.csv")
    print(" ", out_dir / "per_size_deltaI.csv")
    print(" ", out_dir / "per_size_snr_db.csv")
    return df_all

In [ ]:
df_all = batch_size_dependency(
    tsv_glob=r"data\MFE_pro_size\*.tsv",
    out_dir=r"data\MFE_pro_size\size_dep",
    N_on_s=2,  # если N известно — ОБЯЗАТЕЛЬНО лучше фиксировать
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def snr_vs_radius_plot(
    csv_path: str,
    *,
    radius_col: str = "Radius",   # <-- поменяй, если у тебя иначе (например "radius_um")
    snr_col: str = "snr_db",      # или "snr" (линейный)
    out_png: str = "snr_vs_radius.png",
    aggregate_by_unique_radius: bool = True,
):
    df = pd.read_csv(csv_path)

    if radius_col not in df.columns:
        raise RuntimeError(f"Column '{radius_col}' not found. Available: {list(df.columns)[:20]} ...")
    if snr_col not in df.columns:
        raise RuntimeError(f"Column '{snr_col}' not found. Available: {list(df.columns)[:20]} ...")

    r = pd.to_numeric(df[radius_col], errors="coerce").to_numpy(np.float64)
    s = pd.to_numeric(df[snr_col], errors="coerce").to_numpy(np.float64)

    m = np.isfinite(r) & np.isfinite(s)
    r = r[m]; s = s[m]
    if r.size == 0:
        raise RuntimeError("No valid (Radius, SNR) pairs after filtering NaNs.")

    # ---- 1) scatter ----
    plt.figure(figsize=(8,5))
    plt.scatter(r, s, s=10)
    plt.xlabel(radius_col)
    plt.ylabel(snr_col)
    plt.title(f"{snr_col} vs {radius_col} (all droplets)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    Path(out_png).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png)

    # ---- 2) median ± IQR by radius (если радиусы заданы дискретно) ----
    if aggregate_by_unique_radius:
        df2 = pd.DataFrame({radius_col: r, snr_col: s})
        # если радиусы не целые, но "почти целые" — можно округлить:
        # df2[radius_col] = df2[radius_col].round(3)

        grp = df2.groupby(radius_col)[snr_col]
        stats = grp.agg(
            n="count",
            median="median",
            p25=lambda x: np.quantile(x, 0.25),
            p75=lambda x: np.quantile(x, 0.75),
        ).reset_index()

        stats = stats.sort_values(radius_col)
        x = stats[radius_col].to_numpy(np.float64)
        y = stats["median"].to_numpy(np.float64)
        yerr_lo = y - stats["p25"].to_numpy(np.float64)
        yerr_hi = stats["p75"].to_numpy(np.float64) - y

        out_png2 = str(Path(out_png).with_name(Path(out_png).stem + "_median_IQR.png"))

        plt.figure(figsize=(8,5))
        plt.errorbar(x, y, yerr=[yerr_lo, yerr_hi], fmt="o-")
        plt.xlabel(radius_col)
        plt.ylabel(snr_col)
        plt.title(f"{snr_col} vs {radius_col} (median ± IQR)")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(out_png2, dpi=100, bbox_inches="tight")
        plt.show()
        print("Saved:", out_png2)

        # печать таблицы агрегации
        print("\nPer-radius summary:")
        print(stats.to_string(index=False))


# --- пример вызова ---
# snr_vs_radius_plot(
#     csv_path=r"snr_size_analysis/per_droplet.csv",
#     radius_col="Radius_um",  # <-- как у тебя называется
#     snr_col="snr_db",
#     out_png=r"snr_size_analysis/snr_db_vs_radius.png",
# )

In [ ]:

# --- пример вызова ---
snr_vs_radius_plot(
    csv_path=r"data\MFE_pro_size\size_dep\per_droplet.csv",
    radius_col="size_um",  
    snr_col="snr_db",
    out_png=r"snr_size_analysis/snr_db_vs_radius.png",
)